# 04 - Secondary sexual dimorphism in growth

The results generated here were used in manuscript section 3.6. Sex carried
only 2-10 % of the permutation importance
on trait *intensity* (Figure 5), so dimorphism is tested here as a difference
in how each sex *responds* to the environment, through four complementary
analyses:

| Step | Question | Output |
|---|---|---|
| 1 | Can a model tell females from males, and from what? | Table 3 |
| 2 | Do the two sexes have different driver hierarchies? | Table 4 |
| 3 | Which light signals does each sex track more strongly? | Figure 6 |
| 4 | How tightly does each sex's growth follow the light? | Supplementary Table 2 |
| 5 | Do the sexes differ in mean monthly growth? | Supplementary Table 3 |

**Produces:** Table 3, Table 4, Figure 6, Supplementary Table 2,
Supplementary Table 3.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, pearsonr

from yerbamate import config as C
from yerbamate import models as M, plotting as P, stats as S

P.use_paper_style()
pd.set_option("display.width", 200, "display.max_columns", 40)

df = pd.read_csv(C.UNIFIED)
df["system"] = np.where(df.environment == "MO", "MO", "AFS")
print(df.groupby(["system", "sex"]).size().rename("records").to_string())

## Table 3 - GBM sex classifier

One record per axis (n = 90): the five growth traits averaged over the 25
months, plus the environmental features and cultivation system. Leave-one-axis-
out cross-validation, because 90 axes are too few for a held-out split.

The result is the informative part. Only the growth traits carry any
importance; every environmental feature and the system flag land at exactly
zero. Light was measured at plot level, so every plant within a system saw the
same conditions regardless of sex - there is no sex-linked environmental signal
for the model to exploit. The classifier therefore diagnoses sex purely from
*how each shoot grew*.

In [ ]:
axis_level = (df.groupby("axis_id")
                .agg(sex=("sex", "first"),
                     **{v: (v, "mean") for v in C.SEX_CLF_FEATURES})
                .reset_index().dropna(subset=C.SEX_CLF_FEATURES))
X = axis_level[C.SEX_CLF_FEATURES].values
y = (axis_level["sex"] == "M").astype(int).values

accuracy, auc, raw_importance = M.gbm_loo_classifier(X, y)
print(f"leave-one-axis-out accuracy = {accuracy:.1%}   ROC-AUC = {auc:.3f}   n = {len(y)}")

table_3 = pd.DataFrame({"Predictor": C.SEX_CLF_FEATURES, "Importance": raw_importance})
table_3["Importance_pct"] = (table_3["Importance"].clip(lower=0)
                             / table_3["Importance"].clip(lower=0).sum() * 100).round(1)
table_3 = table_3.sort_values("Importance_pct", ascending=False)
table_3["Predictor"] = table_3["Predictor"].map(
    lambda f: C.MORPHO_LABELS.get(f, C.FEATURE_LABELS.get(f, f)))
table_3.to_csv(C.TAB_DIR / "Table_3_sex_classifier_importance.csv", index=False)
print(table_3[["Predictor", "Importance_pct"]].to_string(index=False))

In [ ]:
env_share = table_3[~table_3.Predictor.isin(C.MORPHO_LABELS.values())]["Importance_pct"].sum()
print(f"combined importance of all environmental features and system: {env_share:.1f} %")

## Table 4 - sex-specific driver hierarchies

Separate GBM regressors for female and male axes, on the same 15 predictors
(13 environmental features plus system and rhythm phase; sex is now the
grouping, not a predictor).

The headline is a cross-over on morning R:FR: it is the dominant driver of
female shoot elongation (37 % vs 16 % for males) but of *male* leaf number
increase (30 % vs 2 % for females). The same spectral cue is routed to
different developmental processes in each sex.

In [ ]:
sex_importance = {}
rows = []
for v in C.MORPHO_VARS:
    sex_importance[v] = {}
    for s in ["F", "M"]:
        sub = df[df.sex == s]
        _, imp = M.gbm_regression(sub, C.SEX_REG_FEATURES, v,
                                  model_kwargs=M.SEX_REGRESSOR_KWARGS)
        sex_importance[v][s] = imp
        for f, pct in imp.items():
            rows.append({"Trait": v, "Sex": s, "Feature": f,
                         "Label": C.FEATURE_LABELS[f], "Importance_pct": pct})

sex_fi = pd.DataFrame(rows)
sex_fi.to_csv(C.TAB_DIR / "Table_4_sex_specific_importance_long.csv", index=False)

table_4 = (sex_fi.pivot_table(index="Label", columns=["Trait", "Sex"],
                              values="Importance_pct")
                 .reindex(columns=pd.MultiIndex.from_product([C.MORPHO_VARS, ["F", "M"]])))
table_4 = table_4.reindex(table_4.max(axis=1).sort_values(ascending=False).index)
table_4.round(1).to_csv(C.TAB_DIR / "Table_4_sex_specific_importance.csv")
print(f"n = {(df.sex == 'F').sum()} female and {(df.sex == 'M').sum()} male axis-month records")
print(table_4.round(1).to_string())

## Figure 6 - sex sensitivity to light

For each of the eight light and photoperiod signals and each growth trait, the
female share of permutation importance:

`female sensitivity = female importance / (female importance + male importance)`

Above 0.5 (red) the signal matters more to females, below 0.5 (blue) more to
males. Panel B collapses each row to its mean and plots the distance from
equal tracking.

In [ ]:
sensitivity = pd.DataFrame(
    {v: [M.sex_sensitivity(sex_importance[v]["F"].get(f, 0),
                           sex_importance[v]["M"].get(f, 0))
         for f in C.LIGHT_SIGNALS] for v in C.MORPHO_VARS},
    index=C.LIGHT_SIGNALS).round(3)
sensitivity.index.name = "LightSignal"
sensitivity.to_csv(C.TAB_DIR / "Figure_6_sex_sensitivity.csv")
print(sensitivity.to_string())

In [ ]:
TRAIT_WRAP = ["Shoot\nelongation", "Metamer\nemission", "Leaf no.\nincrease",
              "Leaf area\nincrease", "Leaf\nshed"]
mat = sensitivity[C.MORPHO_VARS].values.astype(float)

fig, axes = plt.subplots(1, 2, figsize=(15.5, 6.2),
                         gridspec_kw={"width_ratios": [1.25, 1]})
im = axes[0].imshow(mat, cmap="RdBu_r", vmin=0, vmax=1, aspect="auto")
axes[0].set_xticks(range(5))
axes[0].set_xticklabels(TRAIT_WRAP, fontsize=12)
axes[0].set_yticks(range(len(C.LIGHT_SIGNALS)))
axes[0].set_yticklabels([C.FEATURE_LABELS[f] for f in C.LIGHT_SIGNALS], fontsize=12.5)
for r in range(mat.shape[0]):
    for c in range(mat.shape[1]):
        axes[0].text(c, r, f"{mat[r, c]:.2f}", ha="center", va="center",
                     fontsize=12, fontweight="bold",
                     color="white" if abs(mat[r, c] - 0.5) > 0.28 else "#222")
cb = fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.03)
cb.set_label("Female share (0 = male-only, 0.5 = equal, 1 = female-only)", fontsize=11.5)
cb.ax.tick_params(labelsize=11)
P.panel(axes[0], "A", x=-0.02, y=1.03, size=20)

mean_share = np.nanmean(mat, axis=1)
deviation = 0.5 - mean_share            # female-dominant rows point left
ypos = np.arange(len(C.LIGHT_SIGNALS))
axes[1].barh(ypos, deviation, alpha=0.9, edgecolor="white",
             color=[P.COL_FEMALE if m > 0.5 else P.COL_MALE for m in mean_share])
axes[1].axvline(0, color="#333", lw=1)
for i, (dev, share) in enumerate(zip(deviation, mean_share)):
    axes[1].text(dev + (0.012 if dev >= 0 else -0.012), i, f"{share:.2f}",
                 va="center", ha="left" if dev >= 0 else "right", fontsize=11.5)
axes[1].set_yticks(ypos)
axes[1].set_yticklabels([C.FEATURE_LABELS[f] for f in C.LIGHT_SIGNALS], fontsize=12.5)
axes[1].set_xlabel("Deviation from equal\n(left = female-dominant, right = male-dominant)",
                   fontsize=12.5)
axes[1].set_xlim(-0.27, 0.36)
P.despine(axes[1])
axes[1].legend(handles=[mpatches.Patch(color=P.COL_FEMALE, label="Female-dominant"),
                        mpatches.Patch(color=P.COL_MALE, label="Male-dominant")],
               loc="upper left", frameon=False, fontsize=11.5)
P.panel(axes[1], "B", x=-0.02, y=1.03, size=20)
fig.tight_layout(w_pad=3)
P.save(fig, "Figure_6")

## Supplementary Table 2 - growth against light, by sex

Pearson correlations over all 25 monthly periods, computed for all axes and
separately for each sex. Shoot elongation is the clearest split: in females it
correlates positively and significantly with all seven light signals, while in
males it is either uncorrelated or slightly negative.

In [ ]:
LIGHT_FOR_CORR = ["PAR_morning", "PAR_midday", "PAR_afternoon",
                  "RFR_morning", "RFR_midday", "RFR_afternoon", "DLI"]
rows = []
for v in C.MORPHO_VARS:
    for group, sub in [("All", df), ("Females", df[df.sex == "F"]), ("Males", df[df.sex == "M"])]:
        for f in LIGHT_FOR_CORR:
            r, p = pearsonr(sub[f], sub[v])
            rows.append({"Variable": C.MORPHO_LABELS[v], "Light_feature": C.FEATURE_LABELS[f],
                         "Group": group, "n": len(sub), "r": round(r, 3),
                         "p": p, "sig": S.stars(p)})

corr = pd.DataFrame(rows)
corr.to_csv(C.TAB_DIR / "Table_S2_growth_light_correlations.csv", index=False)

table_s2 = corr.copy()
table_s2["cell"] = table_s2.apply(lambda r: f"{r.r:+.3f} {r.sig}", axis=1)
table_s2 = table_s2.pivot_table(index=["Variable", "Light_feature"], columns="Group",
                                values="cell", aggfunc="first", sort=False)
print(table_s2[["All", "Females", "Males"]].to_string())

## Supplementary Table 3 - mean monthly growth by sex and system

**A** Mann-Whitney U on each simple contrast. **B** Scheirer-Ray-Hare, the
rank-based two-way test, giving the sex, system and interaction terms.

The apparent dimorphism reversal in elongation - females ahead in monoculture,
males ahead in agroforestry - is a descriptive tendency in the means only.
There is no significant main effect of sex for any trait and no sex x system
interaction. Only the cultivation-system effect is significant.

In [ ]:
means = (df.groupby(["system", "sex"])[C.MORPHO_VARS].mean().round(3)
           .rename(index={"F": "Female", "M": "Male"}))
system_means = df.groupby("system")[C.MORPHO_VARS].mean().round(3)
print("by system and sex:")
print(means.to_string())
print("\nby system:")
print(system_means.to_string())

In [ ]:
rows = []
for v in C.MORPHO_VARS:
    for system in ["MO", "AFS"]:
        f = df[(df.system == system) & (df.sex == "F")][v].dropna()
        m = df[(df.system == system) & (df.sex == "M")][v].dropna()
        U, p = mannwhitneyu(f, m, alternative="two-sided")
        rows.append({"Variable": C.MORPHO_LABELS[v], "Contrast": f"Female vs Male | {system}",
                     "mean_A": round(f.mean(), 2), "mean_B": round(m.mean(), 2),
                     "n_A": len(f), "n_B": len(m), "U": round(float(U)),
                     "p": round(float(p), 4), "sig": S.stars(p)})
    mo = df[df.system == "MO"][v].dropna()
    afs = df[df.system == "AFS"][v].dropna()
    U, p = mannwhitneyu(mo, afs, alternative="two-sided")
    rows.append({"Variable": C.MORPHO_LABELS[v], "Contrast": "MO vs AFS (sexes pooled)",
                 "mean_A": round(mo.mean(), 2), "mean_B": round(afs.mean(), 2),
                 "n_A": len(mo), "n_B": len(afs), "U": round(float(U)),
                 "p": round(float(p), 4), "sig": S.stars(p)})

table_s3a = pd.DataFrame(rows)
table_s3a.to_csv(C.TAB_DIR / "Table_S3A_mannwhitney.csv", index=False)
print(table_s3a.to_string(index=False))

In [ ]:
rows = []
for v in C.MORPHO_VARS:
    res = S.scheirer_ray_hare(df, v, "sex", "system")
    row = {"Variable": C.MORPHO_LABELS[v]}
    for term, out in res.items():
        name = {"sex": "Sex", "system": "System", "sex x system": "Sex x System"}[term]
        row[f"{name} H"] = out["H"]
        row[f"{name} p"] = out["p"]
        row[f"{name} sig"] = out["sig"]
    rows.append(row)

table_s3b = pd.DataFrame(rows)
table_s3b.to_csv(C.TAB_DIR / "Table_S3B_scheirer_ray_hare.csv", index=False)
print(f"n = {len(df):,} axis-month records")
print(table_s3b.to_string(index=False))

In [ ]:
assert abs(accuracy - 0.722) < 0.02 and abs(auc - 0.709) < 0.02
assert abs(table_3.set_index("Predictor").loc["Shoot elongation", "Importance_pct"] - 45.3) < 0.15
assert env_share == 0.0, "environmental features must contribute nothing to sex classification"
assert abs(sex_importance["elongation"]["F"]["RFR_morning"] - 37.0) < 0.15
assert abs(sex_importance["leaf_increase"]["M"]["RFR_morning"] - 30.5) < 0.15
assert abs(sex_importance["metamer_emission"]["F"]["night_hours"] - 72.3) < 0.15
srh = table_s3b.set_index("Variable")
assert (srh["Sex sig"] == "n.s.").all(), "no trait should show a significant sex main effect"
assert (srh["Sex x System sig"] == "n.s.").all()
assert srh.loc["Leaf-area increase (cm2)" if "Leaf-area increase (cm2)" in srh.index
               else "Leaf area increase", "System p"] == 0.0436
print("Validated the Tables 3, 4, S2, S3 and Figure 6 results used in the manuscript.")